In [ ]:
import geemap
import ee 
import os
import pandas as pd
import geopandas as gpd

In [ ]:
geemap.ee_initialize()

In [ ]:
Map = geemap.Map()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')

In [ ]:
# 1. Load and Filter Sentinel-2 Surface Reflectance (2026)
# We use the Harmonized collection for consistency
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

In [ ]:
#Visualising this: 
vis = {
    "min": 0.0,
    "max": 3000,
    "bands": ["B4", "B3", "B2"],
}
Map.centerObject(roi, 10)
Map.addLayer(s2, vis, "Sentinel-2")
Map


Program to Check NDBI 

In the red/orange areas, you are likely looking at residential or commercial zones.

In [ ]:
# NDBI: Built-up/Urban Index
def get_ndbi(image):
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return ndbi

# --- Map 1: NDBI (Built-up / Urban) ---
Map_NDBI = geemap.Map()
Map_NDBI.centerObject(roi, 10)
ndbi_params = {'min': -0.5, 'max': 0.5, 'palette': ['white', 'orange', 'red']}

Map_NDBI.addLayer(get_ndbi(s2), ndbi_params, 'NDBI (Urban)')
Map_NDBI.add_colorbar(vis_params=ndbi_params, label="NDBI Value", orientation="horizontal")
print("Displaying NDBI Map...")
display(Map_NDBI)

Program to check MNDWI 

Look for the deep blue pixels; those will be your lakes, rivers, or reservoirs.

In [ ]:
# MNDWI: Water Index
def get_mndwi(image):
    mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return mndwi

# --- Map 2: MNDWI (Water Index) ---
Map_MNDWI = geemap.Map()
Map_MNDWI.centerObject(roi, 10)
mndwi_params = {'min': -0.5, 'max': 0.5, 'palette': ['white', 'lightblue', 'blue']}

Map_MNDWI.addLayer(get_mndwi(s2), mndwi_params, 'MNDWI (Water)')
Map_MNDWI.add_colorbar(vis_params=mndwi_params, label="MNDWI Value", orientation="horizontal")
print("Displaying MNDWI Map...")
display(Map_MNDWI)

Program to check SAVI

The vibrant green represents healthy biomass. If you compare this to a standard NDVI, you'll notice it handles the edges of fields (where soil is visible) much more accurately.

In [ ]:
# SAVI: Soil Adjusted Vegetation Index
def get_savi(image):
    savi = image.expression(
        '((NIR - RED) * 1.5) / (NIR + RED + 0.5)', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')
    return savi

# --- Map 3: SAVI (Soil Adjusted Vegetation) ---
Map_SAVI = geemap.Map()
Map_SAVI.centerObject(roi, 10)
savi_params = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}

Map_SAVI.addLayer(get_savi(s2), savi_params, 'SAVI (Vegetation)')
Map_SAVI.add_colorbar(vis_params=savi_params, label="SAVI Value", orientation="horizontal")
print("Displaying SAVI Map...")
display(Map_SAVI)

Defining Indexes for Build-up Estimation

In [ ]:
# 2. Define Index Functions
def calculate_indices(image):
    # Band aliases for Sentinel-2
    # B11=SWIR1, B8=NIR, B3=Green, B4=Red
    res = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)', {
            'SWIR1': image.select('B11'),
            'NIR': image.select('B8')
        }).rename('NDBI')
    
    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)', {
            'Green': image.select('B3'),
            'SWIR1': image.select('B11')
        }).rename('MNDWI')
    
    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)', {
            'NIR': image.select('B8'),
            'Red': image.select('B4')
        }).rename('SAVI')
        
    return image.addBands([res, mndwi, savi])

# Apply the indices
processed_img = calculate_indices(s2)

Program to check and fine-tune the logic of Urban Build-up measurement

In [ ]:
# 3. Create the "Clean" Built-up Mask (The Logic)
# Thresholds: 
# NDBI > 0 (Built-up typically positive)
# MNDWI < 0 (Exclude water/moist salt pans)
# SAVI < 0.2 (Exclude vegetation)
built_up_mask = processed_img.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)', {
        'NDBI': processed_img.select('NDBI'),
        'MNDWI': processed_img.select('MNDWI'),
        'SAVI': processed_img.select('SAVI')
    }).rename('BuiltUp')

In [ ]:
# 4. Visualization
Map = geemap.Map()
Map.centerObject(roi, 11)

# True Color for Reference
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2',], 'min': 0, 'max': 3000}, 'S2 True Color (2025)')

# The Result: 1 = White (Built-up), 0 = Black (Everything Else)
Map.addLayer(built_up_mask, {'min': 0, 'max': 1, 'palette': ['black', 'white']}, 'Clean Built-up Layer')

Map

Exporting Image

In [ ]:
# Export the 2025 Built-up Mask to Google Drive
task = ee.batch.Export.image.toDrive(
    image= built_up_mask.toByte(), # Convert to 0-1 byte for smaller file size
    description='Dholera_Builtup_Mask_2025',
    folder='export',
    fileNamePrefix='dholera_builtup_2025',
    region=roi.geometry(),
    scale=10, # Sentinel-2 resolution
    crs='EPSG:32643', # Export in UTM 43N for direct QGIS use
    maxPixels=1e13
)
task.start()
print(" Export started! Check your Google Drive 'Dholera_Project' folder in a few minutes.")

I am providing the geojson file for important_roads, you can access that in data/processed folder. 
But, if you want latest road data, you can use QGIS OSM Pluggin to extract roads: In highway (motorway|trunk|primary|secondary|tertiary) roads are extracted. These are the most important roads in our analysis ensuring we utlise the most important roads only. 

In [ ]:
# --- STEP 1: Local Infrastructure Ingestion ---
# Path to your manually verified roads from QGIS
roads_path = os.path.join('..', 'data', 'Processed', 'important_roads.geojson')

if os.path.exists(roads_path):
    print(f" Loading verified infrastructure from: {roads_path}")
    
    # 1. Load the local GeoJSON into a GeoDataFrame
    major_roads_gdf = gpd.read_file(roads_path)
    
    # Ensure it is in EPSG:4326 before sending to GEE
    if major_roads_gdf.crs != "EPSG:4326":
        major_roads_gdf = major_roads_gdf.to_crs("EPSG:4326")
    
    # 2. Upload to GEE Cloud
    ee_roads = geemap.geopandas_to_ee(major_roads_gdf)
    print(f" {len(major_roads_gdf)} road segments uploaded to GEE.")
else:
    print(" ERROR: File not found. Please check the path.")

In [ ]:
# --- STEP 1.1: Visualize Uploaded Infrastructure ---

# 1. Create the Map
Map_Roads = geemap.Map()

# 2. Define visualization parameters for the lines
road_vis = {
    'color': 'FF0000', # Bright Red
    'width': 2         # Thickness of the line
}

# 3. Add the layer and center the map on the roads
Map_Roads.addLayer(ee_roads, road_vis, 'Verified Important Roads')
Map_Roads.centerObject(ee_roads, zoom=12)

# 4. Display the map
Map_Roads

In [ ]:
# --- STEP 2: GEE Raster Preparation ---

# 1. Distance Raster (Threshold at 20km)
# The distance is calculated in the cloud based on the geometry of ee_roads
dist_raster = ee_roads.distance(20000).rename('dist_m')

# 2. Built-up Density (Neighborhood scale 250m)
# .toFloat() ensures we get decimal density gradients (0.0 to 1.0)
density_raster = built_up_mask.toFloat().focal_mean(radius=250, units='meters').rename('builtup_density')

# 3. Stack Bands into a single 'Analysis Image'
data_stack = dist_raster.addBands(density_raster)

In [ ]:
# --- STEP 3: Sampling & Export ---

# 1. Generate 2000 Random "Survey" Points inside the ROI
sample_points = ee.FeatureCollection.randomPoints(region=roi, points=2000, seed=42)

# 2. ReduceRegions: The "Engine Room"
# We specify the CRS here to ensure 'dist_m' is interpreted as meters
sampled_fc = data_stack.reduceRegions(
    collection=sample_points,
    reducer=ee.Reducer.first(),
    scale=10,
    crs='EPSG:32643'
)

In [ ]:
'''# 3. Convert results to Pandas Dataframe
print("Pulling sampling results into Pandas...")
df_2025 = geemap.ee_to_df(sampled_fc)'''

In [ ]:
print(" Extracting data and coordinates...")

# 1. Get the raw data (which includes geometry)
raw_data = sampled_fc.getInfo()['features']

# 2. Extract properties AND coordinates manually
data_list = []
for f in raw_data:
    props = f['properties']
    # Grab coordinates from the geometry key
    props['longitude'] = f['geometry']['coordinates'][0]
    props['latitude'] = f['geometry']['coordinates'][1]
    data_list.append(props)

# 3. Create DataFrame
df_2025 = pd.DataFrame(data_list)

In [ ]:
# 4. Final Cleaning
# Remove points that might have landed in 'No Data' zones
df_2025 = df_2025.dropna(subset=['dist_m', 'builtup_density'])

print(f"Success! Final dataset contains {len(df_2025)} points.")
print(df_2025.head())

In [ ]:
# Map 1: All Points
center_lat, center_lon = df_2025['latitude'].mean(), df_2025['longitude'].mean()
Map1 = geemap.Map(center=[center_lat, center_lon], zoom=11)
ee_all = geemap.pandas_to_ee(df_2025, latitude='latitude', longitude='longitude')
Map1.addLayer(ee_all, {'color': 'blue'}, 'All Points')
Map1

In [ ]:
# Map 2: Nearest 100
nearest_100 = df_2025.sort_values(by='dist_m').head(100)
Map2 = geemap.Map(center=[center_lat, center_lon], zoom=11)
ee_near = geemap.pandas_to_ee(nearest_100, latitude='latitude', longitude='longitude')
Map2.addLayer(ee_near, {'color': 'red'}, 'Nearest 100 to Roads')

print("✅ Done! Map 1 (All) and Map 2 (Nearest 100) are ready.")
Map2 # Display the nearest 100 map

In [ ]:
# 1. Sort by builtup_density (descending = True for highest first)
top_density_df = df_2025.sort_values(by='builtup_density', ascending=False).head(300)

# 2. Convert this subset to a GEE object
ee_hotspots = geemap.pandas_to_ee(top_density_df, latitude='latitude', longitude='longitude')

# 3. Create the Map
Map = geemap.Map()
Map.centerObject(roi, 11)

# --- Layer 1: True Color Satellite Imagery ---
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'S2 True Color (2025)')

# --- Layer 2: The Binary Mask (Semi-transparent) ---
# We use opacity=0.6 so we can see the satellite imagery underneath
Map.addLayer(built_up_mask, {'min': 0, 'max': 1, 'palette': ['black', 'white']}, 'Clean Built-up Layer', opacity=0.6)

# --- Layer 3: The Hotspots (Top 100 Density) ---
# Using a bright color like 'yellow' or 'cyan' to pop against the white mask
Map.addLayer(ee_hotspots, {'color': 'cyan'}, 'Top 100 Urban Hotspots')

# 4. Print the highest density found to check the data
print(f"Max density found: {top_density_df['builtup_density'].max():.4f}")

Map

In [ ]:
# --- STEP: ADD COORDINATES & EXPORT ---

# 1. Function to pull Lat/Lon out of the geometry and into properties (columns)
def add_lat_lon(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'longitude': coords.get(0),
        'latitude': coords.get(1)
    })

# 2. Map the function over your collection
final_export_fc = sampled_fc.map(add_lat_lon)

# 3. Start the Export Task
task = ee.batch.Export.table.toDrive(
    collection=final_export_fc,
    description='Dholera_Sampling_2025_Final',
    folder='export',
    fileNamePrefix='dholera_samples_with_coords',
    fileFormat='CSV',
    selectors=['latitude', 'longitude', 'dist_m', 'builtup_density'] # Define column order
)

task.start()

print("Export Task started!")